# PayGuard — Model 1: Logistic Regression (Baseline)

Supervised, linear model. This is our **baseline** — every other model should be judged against whether it actually beats this simple, fully interpretable approach.

Trained on all three preprocessing variants (SMOTE, undersampling, class-weighting) so we can see which preprocessing choice helps this model most, then compared later against the other four models.

In [1]:
import joblib
import json
from sklearn.linear_model import LogisticRegression
from evaluation import evaluate_model, print_comparison_table

bundle = joblib.load("../data/processed/train_test_variants.pkl")

X_train_scaled = bundle["X_train_scaled"]
y_train = bundle["y_train"]
X_test_scaled = bundle["X_test_scaled"]
y_test = bundle["y_test"]
class_weight_dict = bundle["class_weight_dict"]
X_train_smote, y_train_smote = bundle["X_train_smote"], bundle["y_train_smote"]
X_train_under, y_train_under = bundle["X_train_under"], bundle["y_train_under"]

print("Bundle loaded. Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)

Bundle loaded. Train shape: (64000, 21) | Test shape: (16000, 21)


## Variant 1: Class-Weighted (trained on original, unresampled data)

In [2]:
lr_weighted = LogisticRegression(class_weight=class_weight_dict, max_iter=1000, random_state=42)
lr_weighted.fit(X_train_scaled, y_train)

y_pred_weighted = lr_weighted.predict(X_test_scaled)
y_score_weighted = lr_weighted.predict_proba(X_test_scaled)[:, 1]

result_weighted = evaluate_model(y_test, y_pred_weighted, "Logistic Regression", "Class-Weighted", y_score_weighted)


  Logistic Regression (Class-Weighted)
Precision: 0.0399   Recall: 0.8222   F1: 0.0761

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15065                 890
Actual Fraud                   8                  37

ROC-AUC: 0.9624

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.94      0.97     15955
       Fraud       0.04      0.82      0.08        45

    accuracy                           0.94     16000
   macro avg       0.52      0.88      0.52     16000
weighted avg       1.00      0.94      0.97     16000



## Variant 2: SMOTE

In [3]:
lr_smote = LogisticRegression(max_iter=1000, random_state=42)
lr_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = lr_smote.predict(X_test_scaled)
y_score_smote = lr_smote.predict_proba(X_test_scaled)[:, 1]

result_smote = evaluate_model(y_test, y_pred_smote, "Logistic Regression", "SMOTE", y_score_smote)


  Logistic Regression (SMOTE)
Precision: 0.0382   Recall: 0.7778   F1: 0.0728

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15074                 881
Actual Fraud                  10                  35

ROC-AUC: 0.9480

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.94      0.97     15955
       Fraud       0.04      0.78      0.07        45

    accuracy                           0.94     16000
   macro avg       0.52      0.86      0.52     16000
weighted avg       1.00      0.94      0.97     16000



## Variant 3: Random Undersampling

In [4]:
lr_under = LogisticRegression(max_iter=1000, random_state=42)
lr_under.fit(X_train_under, y_train_under)

y_pred_under = lr_under.predict(X_test_scaled)
y_score_under = lr_under.predict_proba(X_test_scaled)[:, 1]

result_under = evaluate_model(y_test, y_pred_under, "Logistic Regression", "Undersampling", y_score_under)


  Logistic Regression (Undersampling)
Precision: 0.0313   Recall: 0.8667   F1: 0.0604

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           14747                1208
Actual Fraud                   6                  39

ROC-AUC: 0.9497

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.92      0.96     15955
       Fraud       0.03      0.87      0.06        45

    accuracy                           0.92     16000
   macro avg       0.52      0.90      0.51     16000
weighted avg       1.00      0.92      0.96     16000



## Compare the three preprocessing variants for this model

In [5]:
lr_results = [result_weighted, result_smote, result_under]
lr_comparison_df = print_comparison_table(lr_results)


  MODEL COMPARISON (sorted by F1)
              model        variant  precision  recall     f1  roc_auc  true_positives  false_positives  true_negatives  false_negatives
Logistic Regression Class-Weighted     0.0399  0.8222 0.0761   0.9624              37              890           15065                8
Logistic Regression          SMOTE     0.0382  0.7778 0.0728   0.9480              35              881           15074               10
Logistic Regression  Undersampling     0.0313  0.8667 0.0604   0.9497              39             1208           14747                6


## Interpret the coefficients (the whole point of using Logistic Regression as a baseline)

Unlike the ensemble models, every coefficient here is directly readable — a positive coefficient pushes the prediction toward "fraud," a negative one pushes toward "not fraud." Worth including in your report as a sanity check: do the top features make intuitive sense?

In [6]:
import pandas as pd

coef_df = pd.DataFrame({
    "feature": X_train_scaled.columns,
    "coefficient": lr_weighted.coef_[0],
}).sort_values("coefficient", key=abs, ascending=False)

coef_df

,feature,coefficient
12,sender_balance_error,11.825846
13,receiver_balance_delta,-10.816581
0,amount,8.981194
8,amount_deviation_ratio,8.901824
18,mode_DEBIT,-3.152803
3,sender_balance_after,-2.617369
2,sender_balance_before,2.423026
5,receiver_balance_after,-1.672190
4,receiver_balance_before,1.097400
7,sender_avg_amount_so_far,0.594651


## Save the best variant's model + results for the final 5-model comparison

In [7]:
import os
os.makedirs("../models", exist_ok=True)

# Select the variant with the highest F1 (our selection metric).
# Raw recall alone can reward models that generate too many false alarms.
best_variant = lr_comparison_df.loc[
    lr_comparison_df["f1"].idxmax()
]

print(
    f"Best variant for Logistic Regression: "
    f"{best_variant['variant']} "
    f"(F1={best_variant['f1']}, recall={best_variant['recall']})"
)


variant_to_model = {"Class-Weighted": lr_weighted, "SMOTE": lr_smote, "Undersampling": lr_under}
best_model = variant_to_model[best_variant["variant"]]

joblib.dump(best_model, "../models/logistic_regression.pkl")
lr_comparison_df.to_csv("../models/logistic_regression_results.csv", index=False)
print("Saved model to ../models/logistic_regression.pkl")
print("Saved results to ../models/logistic_regression_results.csv")

Best variant for Logistic Regression: Class-Weighted (F1=0.0761, recall=0.8222)
Saved model to ../models/logistic_regression.pkl
Saved results to ../models/logistic_regression_results.csv


## Next: `04b_model_random_forest.ipynb`